In [1]:
import numpy as np
from scipy.stats import unitary_group
import rotoptsynth as ros
import pennylane as qml

In [5]:
n = 5
N = 2**n
wires = range(n)
dim = 4**n

U = unitary_group.rvs(N, random_state=81512)
ops = ros.rot_opt_synth(U, wires=wires)
print(f"Matrix reproduced correctly: {np.allclose(ros.ops_to_mat(ops, wires), U)}")
num_rotation_angles = ros.count_rotation_angles(ops)
print(f"Number of rotation angles ({num_rotation_angles}) matches group dimension ({dim}): {num_rotation_angles==dim}")

Matrix reproduced correctly: True
Number of rotation angles (1024) matches group dimension (1024): True


In [12]:
ros.enable_validation()
print(qml.draw(ros.rot_opt_synth, wire_order=wires, show_matrices=False)(U, wires=wires))

0: ─────────────────────────────────────────────╭GlobalPhase(-0.09)─╭SelectPauliRot(M4) ···
1: ──U(M0)─╭X──RZ(-0.09)─╭●───────────╭X──U(M2)─├GlobalPhase(-0.09)─├SelectPauliRot(M4) ···
2: ──U(M1)─╰●──RY(0.85)──╰X──RY(1.48)─╰●──U(M3)─╰GlobalPhase(-0.09)─╰SelectPauliRot(M4) ···

0: ··· ───────────────────────────────────────────────────────────────────╭SelectPauliRot(M5) ···
1: ··· ──RY(2.20)──RZ(0.44)─╭●──RX(1.44)─╭●──RZ(9.58)──RY(1.14)──RZ(2.24)─├SelectPauliRot(M5) ···
2: ··· ──RY(2.56)──RZ(1.03)─╰X──RZ(0.64)─╰X──RZ(8.94)──RY(2.65)──RZ(0.51)─╰SelectPauliRot(M5) ···

0: ··· ─╭SelectPauliRot(M6)─╭SelectPauliRot(M7)─╭SelectPauliRot(M8)─╭SelectPauliRot(M9) ···
1: ··· ─├SelectPauliRot(M6)─╰SelectPauliRot(M7)─╰SelectPauliRot(M8)─│────────────────── ···
2: ··· ─╰SelectPauliRot(M6)─────────────────────────────────────────╰SelectPauliRot(M9) ···

0: ··· ─╭SelectPauliRot(M10)────╭SelectPauliRot(M11)─╭SelectPauliRot(M12)─── ···
1: ··· ─│────────────────────╭●─╰SelectPauliRot(M11)─│────────────────

In [4]:
n = 2
N = 2**n
wires = range(n)

U = unitary_group.rvs(N, random_state=81512)
print(qml.draw(ros.asymmetric_two_qubit_decomp, show_matrices=False)(U, wires=wires))

0: ───────────╭●──U(M0)─╭●──RX(1.36)─╭●──U(M2)─╭GlobalPhase(-0.52)─┤  
1: ──RZ(2.00)─╰X──U(M1)─╰X──RZ(0.97)─╰X──U(M3)─╰GlobalPhase(-0.52)─┤  


In [6]:
from rotoptsynth.diag_decomps import _diag_decomp_two_qubits
n = 2
N = 2**n
wires = range(n)

U = unitary_group.rvs(N, random_state=81512)
print(qml.draw(_diag_decomp_two_qubits, show_matrices=False)(U, wires=wires))

0: ─╭U(M0)──RY(0.92)──RZ(1.63)─╭●──RX(1.31)─╭●──RZ(9.72)──RY(0.77)──RZ(1.15)─┤  
1: ─╰U(M0)──RY(1.14)──RZ(0.61)─╰X──RZ(0.81)─╰X──RZ(1.69)──RY(0.71)──RZ(3.99)─┤  


In [7]:
ros.attach_multiplexer_node([qml.RY(0.6, 0)], [qml.RY(-0.1, 0)], "mpx")

[SelectPauliRot(array([ 0.6, -0.1]), wires=['mpx', 0])]

In [8]:
ops0 = [qml.SelectPauliRot([0.6, 0.7], [0], target_wire="target", rot_axis="X")]
ops1 = [qml.SelectPauliRot([0.2, -0.5], [0], target_wire="target", rot_axis="X")]
ros.attach_multiplexer_node(ops0, ops1, "new mpx")

[SelectPauliRot(array([ 0.6,  0.7,  0.2, -0.5]), wires=['new mpx', 0, 'target'])]

In [9]:
ros.attach_multiplexer_node([qml.CNOT([0, 1])], [qml.CNOT([0, 1])], "mpx")

[CNOT(wires=[0, 1])]